# PCAによる画像異常検知：最初に学ぶ再構成誤差ベースの異常検知

このノートブックでは、**PCA（主成分分析）** を使って、画像の異常検知を行います。

異常検知とは、簡単に言えば、

> 多くの正常データとは少し違うデータを見つけること

です。

ここではFashion-MNISTという衣類画像データを使い、

- 正常画像：Sneaker
- 異常画像：Ankle boot

として、**正常画像だけを使ってPCAを学習**します。

その後、正常画像と異常画像をPCAで復元し、

> 元画像と復元画像の差が大きい画像を異常とみなす

という方法で異常検知を行います。

## この実習の狙い

この実習では、いきなり深層学習を使うのではなく、まずは基本的な数理手法であるPCAを使います。

PCAを使うことで、次の流れを理解します。

```text
画像
↓
数値ベクトルに変換
↓
PCAで少数の特徴に圧縮
↓
圧縮した特徴から画像を復元
↓
元画像と復元画像の差を計算
↓
差が大きければ異常と判定
```

この流れは、後で学ぶニューラルネットワークによる画像異常検知の基礎になります。

## 重要な考え方

PCAは、学習データに共通する「変化のパターン」を見つけます。

今回は正常画像であるSneakerだけを使ってPCAを学習するため、PCAは主に

> Sneakerらしい画像の変化

を学びます。

そのため、Sneaker画像は比較的うまく復元できます。一方、Ankle boot画像はSneakerとは形が異なるため、PCAで復元しにくく、元画像との差が大きくなりやすくなります。

この「復元しにくさ」を異常スコアとして使います。


## 1. Google Driveを接続する

Colabの実行環境は一時的なものです。ランタイムを切ると、保存していないファイルは消えます。

そこで、学習結果や図を残すためにGoogle Driveを接続します。


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. 保存先フォルダを作成する

Google Drive内に、この実習用のフォルダを作成します。ここには、後でPCAモデルや評価結果を保存します。


In [ ]:
import os

PROJECT_DIR = '/content/drive/MyDrive/medical_ai_pca_anomaly_fashionmnist'
os.makedirs(PROJECT_DIR, exist_ok=True)

print('保存先フォルダ:', PROJECT_DIR)

## 3. 必要なライブラリを読み込む

今回はPyTorchでFashion-MNISTをダウンロードしますが、モデル学習にはニューラルネットワークを使いません。PCAにはscikit-learnを使います。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from torchvision import datasets, transforms

from sklearn.decomposition import PCA
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

import joblib

%matplotlib inline

np.random.seed(42)
torch.manual_seed(42)

print('NumPy version:', np.__version__)
print('PyTorch version:', torch.__version__)

## 4. Fashion-MNISTをダウンロードする

Fashion-MNISTは、10種類の衣類画像からなるデータセットです。画像は28×28画素のグレースケール画像です。

今回は、次の2クラスだけを使います。

| クラス番号 | 名前 | 役割 |
|---:|---|---|
| 7 | Sneaker | 正常 |
| 9 | Ankle boot | 異常 |


In [ ]:
# Fashion-MNISTの画像をTensorに変換します。
# ToTensor()により、画素値は0〜255ではなく0〜1の範囲になります。
transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

print('学習用データ数:', len(train_dataset))
print('テスト用データ数:', len(test_dataset))

## 5. ラベル名を定義する

Fashion-MNISTのラベル番号を、わかりやすい名前に変換するためのリストを作ります。


In [ ]:
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

NORMAL_CLASS = 7   # Sneaker
ANOMALY_CLASS = 9  # Ankle boot

print('正常クラス:', NORMAL_CLASS, class_names[NORMAL_CLASS])
print('異常クラス:', ANOMALY_CLASS, class_names[ANOMALY_CLASS])

## 6. データセットから画像とラベルを取り出す

PyTorchのDatasetは、1枚ずつ画像を取り出す形式です。PCAでは、まとめてNumPy配列として扱いたいので、画像とラベルをまとめて取り出す関数を作ります。


In [ ]:
def dataset_to_numpy(dataset):
    # PyTorch DatasetをNumPy配列に変換します。
    # imagesの形は (データ数, 1, 28, 28) になります。
    # labelsの形は (データ数,) になります。
    images = []
    labels = []

    for img, label in dataset:
        images.append(img.numpy())
        labels.append(label)

    images = np.stack(images)
    labels = np.array(labels)
    return images, labels

train_images_all, train_labels_all = dataset_to_numpy(train_dataset)
test_images_all, test_labels_all = dataset_to_numpy(test_dataset)

print('train_images_all:', train_images_all.shape)
print('train_labels_all:', train_labels_all.shape)
print('test_images_all:', test_images_all.shape)
print('test_labels_all:', test_labels_all.shape)

## 7. SneakerとAnkle bootだけを取り出す

異常検知では、学習時には **正常画像だけ** を使います。ただし、評価時には正常画像と異常画像の両方を使います。

ここでは、

- 学習用：Sneakerだけ
- テスト用：SneakerとAnkle boot

を作ります。


In [ ]:
# 学習用データから正常クラスだけを取り出します。
train_normal_mask = (train_labels_all == NORMAL_CLASS)
train_normal_images = train_images_all[train_normal_mask]

# テスト用データから正常クラスと異常クラスだけを取り出します。
test_mask = (test_labels_all == NORMAL_CLASS) | (test_labels_all == ANOMALY_CLASS)
test_images = test_images_all[test_mask]
test_original_labels = test_labels_all[test_mask]

# 異常検知用のラベルを作ります。
# 正常 Sneaker は 0、異常 Ankle boot は 1 とします。
test_anomaly_labels = (test_original_labels == ANOMALY_CLASS).astype(int)

print('学習用の正常画像数:', train_normal_images.shape[0])
print('評価用画像数:', test_images.shape[0])
print('評価用 正常数:', np.sum(test_anomaly_labels == 0))
print('評価用 異常数:', np.sum(test_anomaly_labels == 1))

## 8. 画像を表示して確認する

まずは、正常画像と異常画像がどのように違うか目で確認します。


In [ ]:
def show_images(images, labels=None, title='', n=10):
    # 画像を横に並べて表示する関数です。
    plt.figure(figsize=(1.6 * n, 2))
    for i in range(n):
        plt.subplot(1, n, i + 1)
        plt.imshow(images[i, 0], cmap='gray')
        plt.axis('off')
        if labels is not None:
            plt.title(str(labels[i]))
    plt.suptitle(title)
    plt.show()

show_images(train_normal_images[:10], title='Normal class: Sneaker')

abnormal_images_example = test_images[test_anomaly_labels == 1]
show_images(abnormal_images_example[:10], title='Anomaly class: Ankle boot')

## 9. 画像を1次元ベクトルに変換する

PCAは、通常、表形式データに対して使います。画像は28×28の2次元配列ですが、PCAに入力するために、784次元の1次元ベクトルに変換します。

```text
28 × 28 画素 = 784個の数値
```


In [ ]:
def flatten_images(images):
    # 入力 images の形: (データ数, 1, 28, 28)
    # 出力の形: (データ数, 784)
    return images.reshape(images.shape[0], -1)

X_train_normal = flatten_images(train_normal_images)
X_test = flatten_images(test_images)
y_test = test_anomaly_labels

print('PCA学習用データ:', X_train_normal.shape)
print('PCA評価用データ:', X_test.shape)
print('評価ラベル:', y_test.shape)

## 10. PCAとは何をしているのか

PCAは、データのばらつきが大きい方向を見つけ、その方向に沿ってデータを低次元に圧縮する方法です。

画像の場合、784次元の画素情報を、たとえば20個や50個の数値に圧縮します。

この実習では、正常画像だけでPCAを学習します。つまり、PCAは **正常なSneaker画像の代表的な変化の方向** を学習します。

そのため、Sneaker画像はうまく復元しやすく、Ankle boot画像はSneakerらしく復元されて元画像との差が大きくなりやすくなります。


## 11. PCAを正常画像だけで学習する

ここでは主成分数を50にします。主成分数が小さいほど、圧縮が強くなります。

圧縮が強いと画像はぼやけますが、異常検知では「正常らしさだけを残す」効果が出やすくなることがあります。


In [ ]:
N_COMPONENTS = 50

pca = PCA(n_components=N_COMPONENTS, random_state=42)

# 正常画像だけでPCAを学習します。
pca.fit(X_train_normal)

print('PCAの主成分数:', pca.n_components_)
print('累積寄与率:', np.sum(pca.explained_variance_ratio_))

## 12. 主成分の寄与率を確認する

寄与率は、各主成分がどれくらい情報を説明しているかを表します。累積寄与率が高いほど、少数の主成分で元画像の情報を多く保っていることになります。


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o')
plt.xlabel('Number of principal components')
plt.ylabel('Cumulative explained variance ratio')
plt.title('Cumulative explained variance ratio')
plt.grid(True)
plt.show()

## 13. PCAで画像を圧縮して復元する

PCAでは、次の2段階を行います。

1. `transform`：784次元画像を50次元に圧縮する
2. `inverse_transform`：50次元から784次元画像に戻す

この「戻した画像」と元画像の差を、再構成誤差として使います。


In [ ]:
# テスト画像をPCA空間に圧縮します。
Z_test = pca.transform(X_test)

# PCA空間から元の784次元画像に復元します。
X_test_reconstructed = pca.inverse_transform(Z_test)

print('元画像:', X_test.shape)
print('PCAで圧縮した表現:', Z_test.shape)
print('PCAで復元した画像:', X_test_reconstructed.shape)

## 14. 復元画像を表示する

正常画像と異常画像について、元画像、PCAによる復元画像、誤差画像を比べます。


In [ ]:
def show_reconstruction_examples(images, reconstructed, labels, target_label, n=6):
    # 元画像、復元画像、誤差画像を並べて表示します。
    idx = np.where(labels == target_label)[0][:n]

    plt.figure(figsize=(10, 3 * n))

    for row, i in enumerate(idx):
        original_img = images[i].reshape(28, 28)
        recon_img = reconstructed[i].reshape(28, 28)
        error_img = np.abs(original_img - recon_img)

        plt.subplot(n, 3, row * 3 + 1)
        plt.imshow(original_img, cmap='gray')
        plt.title('Original')
        plt.axis('off')

        plt.subplot(n, 3, row * 3 + 2)
        plt.imshow(recon_img, cmap='gray')
        plt.title('PCA reconstruction')
        plt.axis('off')

        plt.subplot(n, 3, row * 3 + 3)
        plt.imshow(error_img, cmap='hot')
        plt.title('Absolute error')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

print('正常画像 Sneaker の復元例')
show_reconstruction_examples(X_test, X_test_reconstructed, y_test, target_label=0, n=6)

print('異常画像 Ankle boot の復元例')
show_reconstruction_examples(X_test, X_test_reconstructed, y_test, target_label=1, n=6)

## 15. 再構成誤差を計算する

元画像と復元画像の差を計算します。ここでは、画像1枚ごとに平均二乗誤差を計算します。

```text
異常スコア = 元画像と復元画像の平均二乗誤差
```

異常スコアが大きいほど、PCAでうまく復元できなかった画像、つまり異常らしい画像と考えます。


In [ ]:
def reconstruction_error_mse(X, X_reconstructed):
    # 画像ごとの平均二乗誤差を計算します。
    errors = np.mean((X - X_reconstructed) ** 2, axis=1)
    return errors

scores = reconstruction_error_mse(X_test, X_test_reconstructed)

print('異常スコアの形:', scores.shape)
print('スコアの最小値:', scores.min())
print('スコアの最大値:', scores.max())
print('スコアの平均値:', scores.mean())

## 16. 正常と異常のスコア分布を比較する

PCAで求めた再構成誤差の分布を、正常画像と異常画像で比較します。

正常と異常の分布が分かれていれば、再構成誤差を使って異常検知ができていることを意味します。

ここでは、

- 正常画像：Sneaker
- 異常画像：Ankle boot

の異常スコアがどの程度分離しているかを見ます。


In [ ]:
normal_scores = scores[y_test == 0]
anomaly_scores = scores[y_test == 1]

plt.figure(figsize=(8, 5))
plt.hist(normal_scores, bins=50, alpha=0.6, label='Normal: Sneaker')
plt.hist(anomaly_scores, bins=50, alpha=0.6, label='Anomaly: Ankle boot')
plt.xlabel('Reconstruction error')
plt.ylabel('Number of images')
plt.title('Distribution of PCA reconstruction errors')
plt.legend()
plt.grid(True)
plt.show()

print('正常画像の平均スコア:', normal_scores.mean())
print('異常画像の平均スコア:', anomaly_scores.mean())

## 17. ROC曲線とAUCを計算する

ROC-AUCは、しきい値をいろいろ変えたときに、正常と異常をどれくらい分けられるかを表す指標です。

- 0.5：ランダムに近い
- 1.0：完全に分離できている


In [ ]:
auc = roc_auc_score(y_test, scores)
print('ROC-AUC:', auc)

fpr, tpr, thresholds = roc_curve(y_test, scores)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f'PCA anomaly detection AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC curve')
plt.legend()
plt.grid(True)
plt.show()

## 18. しきい値を決めて異常判定する

異常スコアは連続値なので、最終的に正常・異常を判定するにはしきい値が必要です。

ここでは、学習に使った正常画像の再構成誤差から、95パーセンタイルをしきい値にします。


In [ ]:
# 学習用正常画像でも復元誤差を計算します。
Z_train_normal = pca.transform(X_train_normal)
X_train_normal_reconstructed = pca.inverse_transform(Z_train_normal)
train_normal_scores = reconstruction_error_mse(X_train_normal, X_train_normal_reconstructed)

# 正常学習データの95パーセンタイルをしきい値にします。
threshold = np.percentile(train_normal_scores, 95)

print('しきい値:', threshold)
print('学習用正常画像の平均スコア:', train_normal_scores.mean())
print('学習用正常画像の95パーセンタイル:', threshold)

## 19. 混同行列で評価する

しきい値より異常スコアが大きい画像を異常と判定します。

```text
score > threshold → 異常
score <= threshold → 正常
```


In [ ]:
y_pred = (scores > threshold).astype(int)
cm = confusion_matrix(y_test, y_pred)

print('混同行列')
print(cm)
print()
print(classification_report(y_test, y_pred, target_names=['Normal Sneaker', 'Anomaly Ankle boot']))

plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.xticks([0, 1], ['Normal', 'Anomaly'])
plt.yticks([0, 1], ['Normal', 'Anomaly'])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center')

plt.colorbar()
plt.show()

## 20. Precision-Recall曲線を見る

異常検知では、異常の見逃しを減らしたいのか、誤検出を減らしたいのかによって、適切なしきい値が変わります。


In [ ]:
precision, recall, pr_thresholds = precision_recall_curve(y_test, scores)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall curve')
plt.grid(True)
plt.show()

## 21. スコアが大きい画像を確認する

異常スコアが大きい画像は、PCAで復元しにくかった画像です。本当にAnkle bootが多いかを確認します。


In [ ]:
def show_top_score_images(images, reconstructed, labels, scores, n=10):
    # np.argsort(scores)[::-1] は逆順になるため、copy()で安全な配列にします。
    idx = np.argsort(scores)[::-1][:n].copy()

    plt.figure(figsize=(1.8 * n, 4))

    for k, i in enumerate(idx):
        original_img = images[i].reshape(28, 28)

        plt.subplot(2, n, k + 1)
        plt.imshow(original_img, cmap='gray')
        plt.title(f'label={labels[i]}
score={scores[i]:.4f}')
        plt.axis('off')

        error_img = np.abs(images[i] - reconstructed[i]).reshape(28, 28)
        plt.subplot(2, n, n + k + 1)
        plt.imshow(error_img, cmap='hot')
        plt.axis('off')

    plt.suptitle('Top anomaly score images: upper=input, lower=error map')
    plt.tight_layout()
    plt.show()

show_top_score_images(X_test, X_test_reconstructed, y_test, scores, n=10)

## 22. スコアが小さい画像を確認する

スコアが小さい画像は、PCAでよく復元できた画像です。Sneakerが多いか確認します。


In [ ]:
def show_low_score_images(images, reconstructed, labels, scores, n=10):
    idx = np.argsort(scores)[:n].copy()

    plt.figure(figsize=(1.8 * n, 4))

    for k, i in enumerate(idx):
        original_img = images[i].reshape(28, 28)

        plt.subplot(2, n, k + 1)
        plt.imshow(original_img, cmap='gray')
        plt.title(f'label={labels[i]}
score={scores[i]:.4f}')
        plt.axis('off')

        error_img = np.abs(images[i] - reconstructed[i]).reshape(28, 28)
        plt.subplot(2, n, n + k + 1)
        plt.imshow(error_img, cmap='hot')
        plt.axis('off')

    plt.suptitle('Lowest anomaly score images: upper=input, lower=error map')
    plt.tight_layout()
    plt.show()

show_low_score_images(X_test, X_test_reconstructed, y_test, scores, n=10)

## 23. 主成分数を変えて実験する

PCAでは、主成分数を変えると結果が変わります。

主成分数が小さいと、情報を強く圧縮します。主成分数が大きいと、元画像をより正確に復元できます。

異常検知では、復元が正確すぎると異常画像も復元できてしまい、異常スコアが下がることがあります。


In [ ]:
component_list = [2, 5, 10, 20, 50, 100, 150]
auc_list = []

for n_comp in component_list:
    pca_tmp = PCA(n_components=n_comp, random_state=42)
    pca_tmp.fit(X_train_normal)

    X_test_recon_tmp = pca_tmp.inverse_transform(pca_tmp.transform(X_test))
    scores_tmp = reconstruction_error_mse(X_test, X_test_recon_tmp)
    auc_tmp = roc_auc_score(y_test, scores_tmp)

    auc_list.append(auc_tmp)
    print(f'n_components={n_comp:3d}, ROC-AUC={auc_tmp:.4f}')

plt.figure(figsize=(7, 4))
plt.plot(component_list, auc_list, marker='o')
plt.xlabel('Number of PCA components')
plt.ylabel('ROC-AUC')
plt.title('Effect of PCA dimension on anomaly detection')
plt.grid(True)
plt.show()

## 24. PCAモデルをGoogle Driveに保存する

学習したPCAモデルをGoogle Driveに保存します。scikit-learnのモデルは、`joblib`を使って保存できます。


In [ ]:
pca_model_path = os.path.join(PROJECT_DIR, 'pca_sneaker_anomaly_model.joblib')

joblib.dump({
    'pca': pca,
    'threshold': threshold,
    'normal_class': NORMAL_CLASS,
    'anomaly_class': ANOMALY_CLASS,
    'class_names': class_names,
    'n_components': N_COMPONENTS
}, pca_model_path)

print('PCAモデルを保存しました:', pca_model_path)

## 25. 保存したPCAモデルを読み込む

保存したモデルを読み込めるか確認します。実際の開発では、学習済みモデルを保存しておくと、別の日に再利用できます。


In [ ]:
loaded = joblib.load(pca_model_path)
loaded_pca = loaded['pca']
loaded_threshold = loaded['threshold']

print('読み込んだ主成分数:', loaded['n_components'])
print('読み込んだしきい値:', loaded_threshold)

## 26. 1枚の画像を判定する関数を作る

最後に、1枚の画像を入力して、異常スコアと正常／異常判定を返す関数を作ります。このように関数化すると、実際のシステムらしくなります。


In [ ]:
def predict_anomaly_pca(image, pca_model, threshold):
    # 1枚の画像についてPCA異常検知を行います。
    # imageの形は (1, 28, 28) または (28, 28) を想定します。

    # 画像を1次元784次元にします。
    x = image.reshape(1, -1)

    # PCAで圧縮して復元します。
    z = pca_model.transform(x)
    x_recon = pca_model.inverse_transform(z)

    # 再構成誤差を計算します。
    score = reconstruction_error_mse(x, x_recon)[0]

    # しきい値より大きければ異常です。
    pred = int(score > threshold)

    reconstructed_image = x_recon.reshape(28, 28)
    return score, pred, reconstructed_image

sample_index = 0
sample_image = test_images[sample_index]
true_label = y_test[sample_index]

score, pred, recon_img = predict_anomaly_pca(sample_image, loaded_pca, loaded_threshold)

print('真のラベル:', true_label, '(0=正常, 1=異常)')
print('異常スコア:', score)
print('予測ラベル:', pred, '(0=正常, 1=異常)')

plt.figure(figsize=(8, 3))

plt.subplot(1, 3, 1)
plt.imshow(sample_image[0], cmap='gray')
plt.title('Input')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(recon_img, cmap='gray')
plt.title('PCA reconstruction')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(np.abs(sample_image[0] - recon_img), cmap='hot')
plt.title('Error map')
plt.axis('off')

plt.tight_layout()
plt.show()

## 27. この実習のまとめ

この実習では、PCAによる画像異常検知を行いました。

重要なポイントは以下です。

1. 画像は数値の集まりとして扱える
2. 28×28画像は784次元ベクトルとして扱える
3. PCAは高次元データを低次元に圧縮する方法である
4. 正常画像だけでPCAを学習すると、正常画像の代表的な変化を学習できる
5. PCAで復元しにくい画像は、正常画像の分布から外れている可能性がある
6. 元画像と復元画像の差を、異常スコアとして使える
7. 異常スコアにしきい値を設定すると、正常／異常の判定ができる

## 次に学ぶ内容へのつながり

今回のPCAでは、画像を

```text
画像 → 少数の特徴 → 復元画像
```

という形で扱いました。

この考え方は、深層学習を使った異常検知にもつながります。

PCAは線形な方法なので、表現できる画像の変化には限界があります。次の段階では、ニューラルネットワークを使って、より柔軟に

```text
画像 → 小さな特徴表現 → 復元画像
```

を学習する方法に進むことができます。

つまり、今回のPCA実習は、画像異常検知の基本構造を理解するための入口です。
